# Hybrid Latent-State Language Model

**Hypothesis:** `latent_state_update() × N` + `decode_token() × M` beats token-by-token.

**Architecture Update:** All models now produce [batch, seq_len, vocab_size] output for fair comparison.
LatentSSM processes tokens sequentially with periodic thinking steps.

| Exp | Model | Steps | Think Every |
|-----|-------|-------|-------------|
| 001 | Transformer baseline | 0 | - |
| 002 | SSM (no thinking) | 0 | - |
| 003 | SSM + thinking | 4 | 4 |
| 004 | SSM + thinking | 4 | 8 |
| 005 | SSM + decoder | 4 | 4 |

In [ ]:
import subprocess, sys, os, json, time, random, math
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Kaggle ships torch 2.10+cu128 which dropped P100 (sm_60) support.
# Install torch 2.3.1+cu118 which still works on P100.
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 7:
    print(f'P100 detected, installing torch 2.3.1+cu118...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
        'torch==2.3.1', 'torchvision==0.18.1'])
    import importlib; importlib.reload(torch)

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, cap: {torch.cuda.get_device_capability()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
@dataclass
class Entity:
    name: str
    location: str
    inventory: List[str] = field(default_factory=list)

class World:
    def __init__(self):
        self.locations = ['kitchen','bedroom','garden','garage','bathroom','office']
        self.items = ['apple','book','key','phone','cup','pen','wallet','watch']
        self.names = ['John','Mary','Alex','Sam','Emma','Leo','Zoe','Max']
        self.entities = {}
        # Guarantee first entity has items
        names_subset = random.sample(self.names, k=random.randint(2,4))
        for i, n in enumerate(names_subset):
            if i == 0:
                inv = random.sample(self.items, k=random.randint(1,2))
            else:
                inv = random.sample(self.items, k=random.randint(0,2))
            self.entities[n] = Entity(name=n, location=random.choice(self.locations), inventory=inv)
    def move(self, name):
        e = self.entities[name]; nl = random.choice([l for l in self.locations if l != e.location])
        ol = e.location; e.location = nl
        return f'{name} moved from {ol} to {nl}.'
    def pickup(self, name):
        e = self.entities[name]; a = [i for i in self.items if i not in e.inventory]
        if not a: return ''
        it = random.choice(a); e.inventory.append(it)
        return f'{name} picked up {it}.'
    def drop(self, name):
        e = self.entities[name]
        if not e.inventory: return ''
        it = random.choice(e.inventory); e.inventory.remove(it)
        return f'{name} dropped {it}.'
    def password(self):
        pw = ''.join(random.choices('ABCDEFGHJKLMNPQRSTUVWXYZ23456789', k=8))
        en = random.choice(list(self.entities.keys()))
        return pw, en, f'The secret code for {en} is {pw}.'

def gen_location():
    w = World(); sents = []
    for n,e in w.entities.items(): sents.append(f'{n} was in the {e.location}.')
    names = list(w.entities.keys())
    for _ in range(5):
        n = random.choice(names); a = random.choice(['move','pickup','drop'])
        if a=='move': sents.append(w.move(n))
        elif a=='pickup':
            r=w.pickup(n)
            if r: sents.append(r)
        else:
            r=w.drop(n)
            if r: sents.append(r)
    for _ in range(3): sents.insert(random.randint(0,len(sents)), random.choice(['It was a quiet day.','The sun was shining.','Time passed slowly.','Nothing unusual happened.']))
    nar = ' '.join(s for s in sents if s)
    t = random.choice(names)
    return nar, f'Where is {t}?', w.entities[t].location

def gen_inventory():
    w = World(); sents = []
    for n,e in w.entities.items():
        inv = ' and '.join(e.inventory) if e.inventory else 'nothing'
        sents.append(f'{n} was in the {e.location} with {inv}.')
    names = list(w.entities.keys())
    for _ in range(4):
        n = random.choice(names); a = random.choice(['move','pickup','drop'])
        if a=='move': sents.append(w.move(n))
        elif a=='pickup':
            r=w.pickup(n)
            if r: sents.append(r)
        else:
            r=w.drop(n)
            if r: sents.append(r)
    nar = ' '.join(s for s in sents if s)
    # Handle case where no one has items
    ts = [n for n in names if w.entities[n].inventory]
    if ts:
        t = random.choice(ts)
        return nar, f'What does {t} have?', ' and '.join(w.entities[t].inventory)
    else:
        t = random.choice(names)
        return nar, f'What does {t} have?', 'nothing'

def gen_recall():
    w = World(); sents = []
    for n,e in w.entities.items(): sents.append(f'{n} was in the {e.location}.')
    pw, en, ps = w.password(); sents.append(ps)
    for _ in range(20): sents.append(random.choice(['It was a quiet day.','The sun was shining brightly.','Time passed slowly.','Nothing unusual happened.','Birds were singing.']))
    return ' '.join(sents), f'What is the secret code for {en}?', pw

def gen_dataset(n=5000, seed=42):
    random.seed(seed); ds = []
    for _ in range(n):
        t = random.choices(['loc','inv','rec','story'], weights=[0.3,0.25,0.25,0.2])[0]
        if t=='loc': nar,q,a = gen_location(); ds.append(dict(narrative=nar,question=q,answer=a,task_type='location'))
        elif t=='inv': nar,q,a = gen_inventory(); ds.append(dict(narrative=nar,question=q,answer=a,task_type='inventory'))
        elif t=='rec': nar,q,a = gen_recall(); ds.append(dict(narrative=nar,question=q,answer=a,task_type='recall'))
        else:
            nm = random.choice(['John','Mary','Alex','Sam','Emma']); s = f'Write a story about {nm}. {nm} was in the kitchen. {nm} looked around.'
            ds.append(dict(narrative=s,question='',answer='And then something happened.',task_type='story'))
    return ds

dataset = gen_dataset()
print(f'Generated {len(dataset)} samples')
print(f'Task dist: {Counter(s["task_type"] for s in dataset)}')

In [ ]:
class Tok:
    def __init__(self, texts, max_vocab=256):
        sp = ['<PAD>','<UNK>','<EOS>','<SEP>']
        c = Counter()
        for t in texts: c.update(t)
        self.vocab = {ch: i+len(sp) for i,(ch,_) in enumerate(c.most_common(max_vocab-len(sp)))}
        for i,t in enumerate(sp): self.vocab[t] = i
        self.inv = {i:c for c,i in self.vocab.items()}
        self.vocab_size = len(self.vocab)
    def encode(self, text, ml=None):
        ids = [self.vocab.get(c, self.vocab['<UNK>']) for c in text]
        if ml: ids = ids[:ml]
        return ids
    def decode(self, ids): return ''.join(self.inv.get(i,'<UNK>') for i in ids)

texts = [f"{s['narrative']} {s['question']} {s['answer']}" for s in dataset if s.get('question')]
tok = Tok(texts, max_vocab=256)
print(f'Vocab: {tok.vocab_size}')

class TD(Dataset):
    def __init__(self, texts, tok, ml=128):
        self.texts = texts; self.tok = tok; self.ml = ml
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        ids = self.tok.encode(self.texts[i], ml=self.ml)
        ids = ids + [self.tok.vocab['<EOS>']]
        ids = ids[:self.ml+1]
        while len(ids) < self.ml+1: ids.append(self.tok.vocab['<PAD>'])
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)

In [ ]:
# ============================================================
# Baseline Transformer
# ============================================================
class PosEnc(nn.Module):
    def __init__(self, d, ml=512, dr=0.1):
        super().__init__(); self.dr = nn.Dropout(dr)
        pe = torch.zeros(ml, d); pos = torch.arange(0,ml,dtype=torch.float).unsqueeze(1)
        dv = torch.exp(torch.arange(0,d,2).float() * (-math.log(10000.0)/d))
        pe[:,0::2] = torch.sin(pos*dv); pe[:,1::2] = torch.cos(pos*dv)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.dr(x + self.pe[:,:x.size(1),:])

class BaselineTransformer(nn.Module):
    def __init__(self, vs, dm=256, nh=8, nl=4, dff=512, dr=0.1, msl=512):
        super().__init__(); self.dm = dm
        self.emb = nn.Embedding(vs, dm); self.pe = PosEnc(dm, msl, dr)
        el = nn.TransformerEncoderLayer(d_model=dm, nhead=nh, dim_feedforward=dff, dropout=dr, batch_first=True)
        self.tr = nn.TransformerEncoder(el, num_layers=nl)
        self.out = nn.Linear(dm, vs)
    def forward(self, src):
        sz = src.size(1)
        mask = torch.triu(torch.ones(sz,sz,device=src.device),1).masked_fill(torch.triu(torch.ones(sz,sz,device=src.device),1)==1, float('-inf'))
        x = self.emb(src) * math.sqrt(self.dm); x = self.pe(x); x = self.tr(x, mask=mask)
        return self.out(x)  # [batch, seq, vocab]

In [ ]:
# ============================================================
# SSM Layer + Latent SSM (Causal LM with thinking)
# ============================================================
class SSMLayer(nn.Module):
    def __init__(self, ds, di, selective=True):
        super().__init__(); self.ds = ds; self.selective = selective
        self.A_base = nn.Parameter(torch.randn(ds,ds)*0.1)
        if selective:
            self.A_mod = nn.Linear(di, ds*ds, bias=False)
            self.B = nn.Linear(di, ds, bias=False)
        else:
            self.A = self.A_base
            self.B = nn.Linear(di, ds)
        self.act = nn.SiLU()
        self.norm = nn.LayerNorm(ds)
    def step(self, x, h):
        if self.selective:
            bs = x.size(0)
            A_delta = self.A_mod(x).view(bs, self.ds, self.ds)
            A = self.A_base.unsqueeze(0) + 0.1*torch.tanh(A_delta)
            h = torch.bmm(A, h.unsqueeze(-1)).squeeze(-1) + self.B(x)
        else:
            h = torch.einsum('ij,bj->bi', self.A_base, h) + self.B(x)
        return self.norm(self.act(h))
        h = torch.einsum('ij,bj->bi', self.A, h) + self.B(x)
        h = self.act(h)
        return self.norm(h)

class LatentSSM(nn.Module):
    """Causal LM with sequential token processing and periodic thinking."""
    def __init__(self, vs, ds=256, dm=256, nsl=2, ls=4, te=4, dr=0.1):
        super().__init__()
        self.ds = ds; self.ls = ls; self.te = te
        self.emb = nn.Embedding(vs, dm)
        self.input_proj = nn.Linear(dm, ds)
        self.ssms = nn.ModuleList([SSMLayer(ds,ds) for _ in range(nsl)])
        self.ffn = nn.Sequential(nn.Linear(ds,ds*2), nn.SiLU(), nn.Dropout(dr), nn.Linear(ds*2,ds))
        self.dec = nn.Linear(ds, vs)
        self.gate = nn.Sequential(nn.Linear(ds,ds), nn.Sigmoid())
    def think(self, h, steps=None):
        for _ in range(steps or self.ls):
            hn = h
            for s in self.ssms: hn = s.step(hn, hn)
            hn = self.ffn(hn); g = self.gate(h); h = g*hn + (1-g)*h
        return h
    def forward(self, src):
        batch, seq_len = src.shape
        h = torch.zeros(batch, self.ds, device=src.device)
        x = self.input_proj(self.emb(src))
        all_logits = []
        for t in range(seq_len):
            for ssm in self.ssms:
                h = ssm.step(x[:,t,:], h)
            if self.te > 0 and (t+1) % self.te == 0:
                h = self.think(h)
            all_logits.append(self.dec(h))
        return torch.stack(all_logits, dim=1)  # [batch, seq, vocab]

In [ ]:
# ============================================================
# Latent SSM + Decoder (multi-token generation)
# ============================================================
class LatentSSMDecoder(nn.Module):
    """SSM + FFN decoder for multi-token generation."""
    def __init__(self, vs, ds=256, dm=256, nsl=2, ls=4, tps=8, te=4, dr=0.1, selective=True):
        super().__init__()
        self.ds = ds; self.ls = ls; self.tps = tps; self.te = te
        self.emb = nn.Embedding(vs, dm)
        self.input_proj = nn.Linear(dm, ds)
        self.ssms = nn.ModuleList([SSMLayer(ds,ds,selective=selective) for _ in range(nsl)])
        self.ffn = nn.Sequential(nn.Linear(ds,ds*2), nn.SiLU(), nn.Dropout(dr), nn.Linear(ds*2,ds))
        self.pos = nn.Parameter(torch.randn(tps,32))
        self.dec = nn.Sequential(nn.Linear(ds+32,ds), nn.SiLU(), nn.Dropout(dr), nn.Linear(ds,vs))
        self.gate = nn.Sequential(nn.Linear(ds,ds), nn.Sigmoid())
    def think(self, h, steps=None):
        for _ in range(steps or self.ls):
            hn = h
            for s in self.ssms: hn = s.step(hn, hn)
            hn = self.ffn(hn); g = self.gate(h); h = g*hn + (1-g)*h
        return h
    def decode_tokens(self, h, n_tokens):
        n = min(n_tokens, self.tps); b = h.size(0)
        hx = h.unsqueeze(1).expand(-1,n,-1); p = self.pos[:n].unsqueeze(0).expand(b,-1,-1)
        return self.dec(torch.cat([hx,p],dim=-1))
    def forward(self, src):
        batch, seq_len = src.shape
        h = torch.zeros(batch, self.ds, device=src.device)
        x = self.input_proj(self.emb(src))
        all_logits = []
        for t in range(seq_len):
            for ssm in self.ssms:
                h = ssm.step(x[:,t,:], h)
            if self.te > 0 and (t+1) % self.te == 0:
                h = self.think(h)
            logits_t = self.decode_tokens(h, 1).squeeze(1)
            all_logits.append(logits_t)
        return torch.stack(all_logits, dim=1)  # [batch, seq, vocab]

print('Models OK - all produce [batch, seq_len, vocab_size]')

In [ ]:
# ============================================================
# Experiments
# ============================================================
EXPS = [
    dict(exp_id='exp001', model='baseline',           ls=0, te=0, dm=256, ep=30, bs=32),
    dict(exp_id='exp002', model='latent_ssm',         ls=0, te=0, dm=256, ep=30, bs=32),  # SSM only, no thinking
    dict(exp_id='exp003', model='latent_ssm',         ls=4, te=4, dm=256, ep=30, bs=32),  # Think every 4 tokens
    dict(exp_id='exp004', model='latent_ssm',         ls=4, te=8, dm=256, ep=30, bs=32),  # Think every 8 tokens
    dict(exp_id='exp005', model='latent_ssm_decoder', ls=4, te=4, dm=256, ep=30, bs=32),  # Decoder variant
]

os.makedirs('experiments', exist_ok=True)
results = {}

for c in EXPS:
    print(f"\n{'='*60}")
    print(f"{c['exp_id']}: {c['model']} (steps={c['ls']}, think_every={c['te']})")
    print(f"{'='*60}")
    tr = dataset[:4000]; va = dataset[4000:]
    tt = [f"{s['narrative']} {s['question']} {s['answer']}" for s in tr if s.get('question')]
    vt = [f"{s['narrative']} {s['question']} {s['answer']}" for s in va if s.get('question')]
    
    # Build model
    if c['model']=='baseline':
        m = BaselineTransformer(vs=tok.vocab_size, dm=c['dm'], nh=8, nl=4)
    elif c['model']=='latent_ssm':
        m = LatentSSM(vs=tok.vocab_size, ds=c['dm'], dm=c['dm'], nsl=2, ls=c['ls'], te=c['te'])
    elif c['model']=='latent_ssm_decoder':
        m = LatentSSMDecoder(vs=tok.vocab_size, ds=c['dm'], dm=c['dm'], nsl=2, ls=c['ls'], tps=8, te=c['te'])
    
    m = m.to(device)
    np_ = sum(p.numel() for p in m.parameters())
    print(f'Params: {np_:,}')
    
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=c['ep']*100, pct_start=0.1)
    trl = DataLoader(TD(tt, tok, ml=128), batch_size=c['bs'], shuffle=True)
    val = DataLoader(TD(vt, tok, ml=128), batch_size=c['bs'])
    
    edir = f"experiments/{c['exp_id']}"
    os.makedirs(edir, exist_ok=True)
    met = {'train_loss':[], 'val_loss':[]}
    
    for ep in range(1, c['ep']+1):
        m.train(); tl = 0
        for bx,by in tqdm(trl, desc=f'  Ep {ep}', leave=False):
            bx,by = bx.to(device), by.to(device)
            opt.zero_grad()
            logits = m(bx)  # All models: [batch, seq, vocab]
            loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)), by.reshape(-1), ignore_index=tok.vocab['<PAD>'])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sch.step()
            tl += loss.item()
        al = tl/len(trl)
        met['train_loss'].append(al)
        
        m.eval(); vl = 0
        with torch.no_grad():
            for bx,by in val:
                bx,by = bx.to(device), by.to(device)
                logits = m(bx)
                loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)), by.reshape(-1), ignore_index=tok.vocab['<PAD>'])
                vl += loss.item()
        vl /= len(val)
        met['val_loss'].append(vl)
        
        if ep%5==0 or ep==c['ep']:
            print(f'  Ep {ep}/{c["ep"]} | train: {al:.4f} | val: {vl:.4f}')
        
        if ep%10==0 or ep==c['ep']:
            torch.save(dict(epoch=ep, model=m.state_dict(), metrics=met), f'{edir}/checkpoint.pt')
    
    torch.save(m.state_dict(), f'{edir}/best_model.pt')
    with open(f'{edir}/metrics.json','w') as f:
        json.dump(met, f, indent=2)
    results[c['exp_id']] = dict(model=c['model'], steps=c['ls'], think_every=c['te'], train_loss=al, val_loss=vl, params=np_)
    print(f'  DONE -> {edir}/')

print(f"\n{'Exp':<10} {'Model':<20} {'Steps':<6} {'TE':<4} {'Params':<10} {'Train':<10} {'Val':<10}")
print('-'*70)
for c in EXPS:
    r = results[c['exp_id']]
    print(f"{c['exp_id']:<10} {r['model']:<20} {r['steps']:<6} {r['think_every']:<4} {r['params']:<10,} {r['train_loss']:<10.4f} {r['val_loss']:<10.4f}")

with open('results.json','w') as f:
    json.dump(results, f, indent=2)
print('\nSaved results.json')

In [ ]:
# ============================================================
# Improved evaluation with temperature sampling
# ============================================================
def generate_sample(model, prompt, tok, device, max_tokens=50, temperature=0.8, top_k=40):
    """Generate text with temperature sampling."""
    model.eval()
    prompt_ids = tok.encode(prompt, ml=128)
    generated = list(prompt_ids)
    
    import torch.nn.functional as F
    with torch.no_grad():
        for _ in range(max_tokens):
            x = torch.tensor([generated], dtype=torch.long).to(device)
            logits = model(x)
            next_logits = logits[0, -1, :] / temperature
            
            # Top-k filtering
            if top_k > 0:
                indices_to_remove = next_logits < torch.topk(next_logits, top_k)[0][..., -1, None]
                next_logits[indices_to_remove] = float('-inf')
            
            # Sample from distribution
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            
            if tok.inv.get(next_token) == '<EOS>':
                break
            generated.append(next_token)
    
    return tok.decode(generated[len(prompt_ids):])

def evaluate_qa(model, dataset, tok, device, n_samples=3):
    """Evaluate QA with multiple samples per question."""
    correct = 0
    total = 0
    by_task = {}
    
    for sample in dataset[:50]:  # Evaluate on subset for speed
        if not sample.get('question'):
            continue
        
        task_type = sample['task_type']
        if task_type not in by_task:
            by_task[task_type] = {'correct': 0, 'total': 0}
        
        prompt = f"{sample['narrative']} {sample['question']}"
        answer = sample['answer']
        
        # Try multiple samples
        found = False
        for _ in range(n_samples):
            gen = generate_sample(model, prompt, tok, device, temperature=0.8, top_k=40)
            if answer.lower() in gen.lower():
                found = True
                break
        
        if found:
            correct += 1
            by_task[task_type]['correct'] += 1
        total += 1
        by_task[task_type]['total'] += 1
    
    task_acc = {t: c['correct']/max(c['total'],1) for t,c in by_task.items()}
    return correct/max(total,1), task_acc

# Evaluate best models
print('\n' + '='*60)
print('Evaluating best models with improved QA eval')
print('='*60)

qa_results = {}
for exp_id in ['exp001', 'exp003', 'exp005']:
    c = next(e for e in EXPS if e['exp_id'] == exp_id)
    print(f"\n--- {exp_id} ({c['model']}) ---")
    
    # Build and load model
    if c['model']=='baseline':
        m = BaselineTransformer(vs=tok.vocab_size, dm=c['dm'], nh=8, nl=4)
    elif c['model']=='latent_ssm':
        m = LatentSSM(vs=tok.vocab_size, ds=c['dm'], dm=c['dm'], nsl=2, ls=c['ls'], te=c['te'])
    elif c['model']=='latent_ssm_decoder':
        m = LatentSSMDecoder(vs=tok.vocab_size, ds=c['dm'], dm=c['dm'], nsl=2, ls=c['ls'], tps=8, te=c['te'])
    
    m.load_state_dict(torch.load(f"experiments/{exp_id}/best_model.pt", map_location=device))
    m = m.to(device)
    
    # Evaluate
    acc, task_acc = evaluate_qa(m, dataset, tok, device, n_samples=3)
    qa_results[exp_id] = {'accuracy': acc, 'task_accuracy': task_acc}
    print(f"Overall accuracy: {acc:.3f}")
    for task, task_a in task_acc.items():
        print(f"  {task}: {task_a:.3f}")
    
    # Generate a few samples
    print(f"\nSample generations:")
    for sample in dataset[:3]:
        if sample.get('question'):
            prompt = f"{sample['narrative']} {sample['question']}"
            gen = generate_sample(m, prompt, tok, device, temperature=0.8)
            print(f"  [{sample['task_type']}] Expected: {sample['answer'][:30]}")
            print(f"  Generated: {gen[:50]}")

with open('qa_results.json','w') as f:
    json.dump(qa_results, f, indent=2)
print('\nSaved qa_results.json')